#### `parse_and_resample.ipnyb`<br>
Sorts on 1:1:1 agreement in gold label annotated data, preserves text and annotation rationales, randomly samples $n$ = 50 if $N_{instances}$ $\geq$ 50 organically. Outputs to `.tsv`.

In [ ]:
import pandas as pd
from pathlib import Path

# Paths
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Label aliases
LABELS = ["asp", "dep", "val", "prg", "tgd", "age", "race", "dbty"]

# Sampling parameters
SAMPLE_SIZE = 50
RANDOM_SEED = 56

In [ ]:
# 1. Load and combine Excel files
df_01 = pd.read_excel(RAW_DIR / "d_cycle999_prp_sd_ss_gpt_agreed_01.xlsx")
df_02 = pd.read_excel(RAW_DIR / "d_cycle999_prp_sd_ss_gpt_agreed_02.xlsx")

df = pd.concat([df_01, df_02], ignore_index=True)
print(f"Combined dataset: {len(df):,} rows, {len(df.columns)} columns")

# Export to Parquet
parquet_path = PROCESSED_DIR / "d_cycle999_combined.parquet"
df.to_parquet(parquet_path, index=False)
print(f"Saved to: {parquet_path}")

In [ ]:
# 2. Filter, sample, and export for each label
for label in LABELS:
    # Column names for this label
    col_ss = f"{label}_ss"
    col_sd = f"{label}_sd"
    col_gpt = f"{label}_gpt"
    col_rtnl = f"{label}_rtnl_gpt"
    
    # Check columns exist
    required_cols = [col_ss, col_sd, col_gpt, col_rtnl]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"[{label}] Skipping - missing columns: {missing}")
        continue
    
    # Filter: all three annotators agree (ss, sd, gpt all = 1)
    mask = (df[col_ss] == 1) & (df[col_sd] == 1) & (df[col_gpt] == 1)
    df_filtered = df.loc[mask, ["p_id_ss", "text", col_rtnl]].copy()
    df_filtered.rename(columns={"p_id_ss": "id"}, inplace=True)
    
    n_total = len(df_filtered)
    
    # Sample if >= SAMPLE_SIZE, otherwise keep all
    if n_total >= SAMPLE_SIZE:
        df_sampled = df_filtered.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)
    else:
        df_sampled = df_filtered
    
    # Export to TSV
    output_path = PROCESSED_DIR / f"{label}_agreed.tsv"
    df_sampled.to_csv(output_path, sep="\t", index=False)
    
    print(f"[{label}] {n_total:,} agreed → {len(df_sampled)} sampled → {output_path.name}")

In [ ]:
# 3. Summary of outputs
print("Generated files:")
for tsv_file in sorted(PROCESSED_DIR.glob("*_agreed.tsv")):
    df_check = pd.read_csv(tsv_file, sep="\t")
    print(f"  {tsv_file.name}: {len(df_check)} rows")